In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("Dados Históricos - Ibovespa2.csv", parse_dates=[0], index_col=0, decimal=',', thousands='.') # Alteração: ajustando a colunda de data e colocando como index 
df.index = pd.to_datetime(df.index, format = "%d.%m.%Y")
df = df.sort_index()

df["y"] = np.sign(df["Último"].diff())
df["y"] = (df["y"] > 0).astype(int)

df = df.dropna()

df["y"].value_counts(normalize=True)

#  Engenharia de Atributos (Feature Engineering)
df["retorno_1"] = df["Último"].pct_change().shift(1)
df["retorno_3"] = df["Último"].pct_change(3).shift(1)

df["vol_5"] = df["Último"].pct_change().rolling(5).std().shift(1)
df["vol_10"] = df["Último"].pct_change().rolling(10).std().shift(1)

# Calculando a diferença da tendencias
df["ma_5"] = df["Último"].rolling(5).mean().shift(1)

df["ma_10"] = df["Último"].rolling(10).mean().shift(1)
df["ma_diff"] = df["ma_5"] - df["ma_10"]

# calculando a diferença de range
df["range_diff"] = (df["Máxima"] - df["Mínima"]).shift(1)

# 1. Distâncias Relativas (Substituindo ma_5 e ma_10 brutos)
df["dist_ma_5"] = ((df["Último"] - df["Último"].rolling(5).mean()) / df["Último"].rolling(5).mean()).shift(1)
df["dist_ma_10"] = ((df["Último"] - df["Último"].rolling(10).mean()) / df["Último"].rolling(10).mean()).shift(1)

# 2. Pressão de Compra/Venda (Onde fechou dentro do candle anterior)
df["pressao_fechamento"] = ((df["Último"] - df["Mínima"]) / (df["Máxima"] - df["Mínima"] + 1e-9)).shift(1)

df["momentum_5"] = (df["Último"] - df["Último"].shift(5)).shift(1)

df.dropna(inplace=True)

test = df.tail(30)
train = df.iloc[:-30]

features = ["retorno_1", "retorno_3", "ma_10", "ma_5", "ma_diff","momentum_5", "range_diff", "dist_ma_5", "dist_ma_10", "pressao_fechamento"]

# definindo x e y de treino
X_train = train[features]
y_train = train["y"]

# definindo x e y de teste
X_test = test[features]
y_test = test["y"]

model_rf = RandomForestClassifier(n_estimators=350, max_depth=10, min_samples_split=2, class_weight="balanced", random_state=42)

model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_rf))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_rf))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_rf))

Accuracy: 0.7333333333333333

Matriz de confusão:  [[10  3]
 [ 5 12]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.67      0.77      0.71        13
           1       0.80      0.71      0.75        17

    accuracy                           0.73        30
   macro avg       0.73      0.74      0.73        30
weighted avg       0.74      0.73      0.73        30

